# Học tổ hợp: Voting và Stacking

*Bài 9, mục 9.2.1.* Kết hợp các mô hình **khác loại** (hồi quy Logistic, SVM, cây quyết định, …).
- **Voting**: bỏ phiếu đa số nhãn (hard) hoặc trung bình xác suất (soft).
- **Stacking**: một **mô hình meta** học cách kết hợp dự đoán của các mô hình cơ sở. Để tránh quá khớp, dự đoán dùng huấn luyện mô hình meta được tạo bằng kiểm định chéo.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

base_models = [('logistic', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
               ('svm', make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(), ensemble=False))),   # SVC kèm ước lượng xác suất
               ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier())),
               ('tree', DecisionTreeClassifier(max_depth=4, random_state=0))]
for name, model in base_models:
    model.fit(X_train, y_train)
    print('%-9s kiểm tra %.4f' % (name, model.score(X_test, y_test)))

logistic  kiểm tra 0.9591
svm       kiểm tra 0.9474


knn       kiểm tra 0.9474
tree      kiểm tra 0.9064


## 1. Voting

In [2]:
from sklearn.ensemble import VotingClassifier

for voting in ['hard', 'soft']:
    ensemble = VotingClassifier(base_models, voting=voting).fit(X_train, y_train)
    print('Voting %-4s: kiểm tra %.4f' % (voting, ensemble.score(X_test, y_test)))

Voting hard: kiểm tra 0.9474
Voting soft: kiểm tra 0.9415


## 2. Stacking

Mô hình meta là hồi quy Logistic, nhận đầu vào là xác suất dự đoán của 4 mô hình cơ sở (tạo bằng kiểm định chéo 5 phần trên tập huấn luyện).

In [3]:
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(), cv=5)
stacking.fit(X_train, y_train)
print('Stacking: kiểm tra %.4f' % stacking.score(X_test, y_test))
print('Trọng số mô hình meta gán cho các mô hình cơ sở:')
for (name, _), coef in zip(base_models, stacking.final_estimator_.coef_[0]):
    print('   %-9s %.3f' % (name, coef))

Stacking: kiểm tra 0.9532
Trọng số mô hình meta gán cho các mô hình cơ sở:
   logistic  2.669
   svm       2.853
   knn       1.825
   tree      1.240
